# Notebook 4 - Ejecucion y Costes
Backtesting LONG-only realista de la estrategia Momentum de Notebook 3.

## Configuracion

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.fs as fs

sns.set(style="whitegrid", context="notebook")

START_DATE = pd.Timestamp("2015-01-01")
INITIAL_CAPITAL = 250000.0
TOP_N = 20
TARGET_W = 1.0 / TOP_N
FEE_RATE = 0.0023
MIN_FEE = 23.0
BENCH = "SPY"

# Control de rotacion: 1=mensual, 3=trimestral, 6=semestral
REBALANCE_EVERY_N_MONTHS = 3

PRICE_PATH_HINTS = [
    "data/processed/prices.parquet",
    "data/raw/sp500_history.parquet",
    "notebooks/data/processed/prices.parquet",
    "notebooks/data/raw/sp500_history.parquet",
    r"C:\Users\alons\Desktop\Pr?ctica 7\sp500_history.parquet",
    r"C:\Users\alons\Desktop\Practica 7\sp500_history.parquet",
]

SELECTION_PATH_HINTS = [
    "outputs/selected_top20_by_rebalance.csv",
    "outputs/selected_20_each_rebalance.csv",
    "notebooks/outputs/selected_top20_by_rebalance.csv",
    "notebooks/outputs/selected_20_each_rebalance.csv",
    "selected_top20_by_rebalance.csv",
]


## Funciones base

In [ ]:
def compute_commission(notional, rate=0.0023, min_fee=23.0):
    n = float(abs(notional))
    if n <= 0:
        return 0.0
    return max(rate * n, min_fee)


def get_month_end_dates(index):
    idx = pd.DatetimeIndex(index).sort_values().unique()
    s = pd.Series(idx, index=idx)
    out = s.groupby(s.index.to_period("M")).max().sort_values()
    return pd.DatetimeIndex(out.values)


def compute_metrics(equity, benchmark=None, freq=252):
    eq = pd.Series(equity).dropna().sort_index()
    if len(eq) < 2:
        return {}
    r = eq.pct_change().dropna()
    yrs = len(r) / float(freq)
    cagr = (eq.iloc[-1] / eq.iloc[0]) ** (1 / yrs) - 1 if yrs > 0 else np.nan
    vol = r.std(ddof=0) * np.sqrt(freq)
    sharpe = (r.mean() / r.std(ddof=0) * np.sqrt(freq)) if r.std(ddof=0) > 0 else np.nan
    dd = eq / eq.cummax() - 1
    out = {
        "total_return": eq.iloc[-1] / eq.iloc[0] - 1,
        "cagr": cagr,
        "vol": vol,
        "sharpe": sharpe,
        "max_drawdown": dd.min(),
    }
    if benchmark is not None:
        b = pd.Series(benchmark).dropna().sort_index()
        a = pd.concat([eq, b], axis=1, join="inner").dropna()
        if len(a) > 2:
            rs = a.iloc[:, 0].pct_change().dropna()
            rb = a.iloc[:, 1].pct_change().dropna()
            x = pd.concat([rs, rb], axis=1).dropna()
            if len(x) > 2:
                out["alpha_daily_mean"] = (x.iloc[:, 0] - x.iloc[:, 1]).mean()
                vb = np.var(x.iloc[:, 1])
                out["beta"] = np.cov(x.iloc[:, 0], x.iloc[:, 1])[0, 1] / vb if vb > 0 else np.nan
    return out


def monte_carlo_monkeys(monthly_returns, n_monos=2000, chunk_size=6, fee_rate=0.0023):
    r = pd.Series(monthly_returns).dropna().astype(float)
    if len(r) < max(24, chunk_size * 2):
        raise ValueError("No hay suficientes retornos mensuales para Monte Carlo por bloques.")
    v = r.values
    n = len(v)
    starts = np.arange(n - chunk_size + 1)
    drag = fee_rate / 12.0
    rows = []
    for _ in range(int(n_monos)):
        path = []
        while len(path) < n:
            st = np.random.choice(starts)
            path.extend(v[st:st + chunk_size].tolist())
        path = np.array(path[:n]) - drag
        wealth = np.cumprod(1 + path)
        yrs = n / 12.0
        cagr = wealth[-1] ** (1 / yrs) - 1 if yrs > 0 and wealth[-1] > 0 else np.nan
        vol = np.std(path, ddof=0) * np.sqrt(12)
        sharpe = (np.mean(path) / np.std(path, ddof=0) * np.sqrt(12)) if np.std(path, ddof=0) > 0 else np.nan
        rows.append((wealth[-1] - 1, cagr, vol, sharpe))
    return pd.DataFrame(rows, columns=["total_return", "cagr", "vol", "sharpe"])

## Carga y adapter de precios (OPEN/CLOSE)

In [ ]:
def _scan_files(roots):
    lfs = fs.LocalFileSystem()
    out, seen = [], set()
    for root in roots:
        try:
            info = lfs.get_file_info(root)
            if info.type == fs.FileType.File:
                if root not in seen:
                    out.append(root)
                    seen.add(root)
                continue
        except Exception:
            pass
        try:
            infos = lfs.get_file_info(fs.FileSelector(root, recursive=True))
        except Exception:
            continue
        for i in infos:
            if i.type != fs.FileType.File:
                continue
            p = i.path
            pl = p.lower()
            if pl.endswith('.parquet') or pl.endswith('.csv'):
                if 'selected_top20_by_rebalance' in pl:
                    continue
                if p not in seen:
                    out.append(p)
                    seen.add(p)
    return out


def _read_any(path):
    try:
        if path.lower().endswith('.parquet'):
            return pq.read_table(path).to_pandas()
        if path.lower().endswith('.csv'):
            return pd.read_csv(path)
    except Exception:
        return None
    return None


def _pick_field(df, axis_level, candidates):
    vals = pd.Index([str(c).strip().lower() for c in df.columns.get_level_values(axis_level)])
    for c in candidates:
        if c in vals.values:
            return c
    return None


def _norm_prices(df):
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return None, None, None

    op, cl, adj = None, None, None

    if isinstance(df.columns, pd.MultiIndex):
        l0 = pd.Index([str(c).strip().lower() for c in df.columns.get_level_values(0)])
        l1 = pd.Index([str(c).strip().lower() for c in df.columns.get_level_values(1)])
        fset = {'open', 'close', 'adj_close', 'adj close', 'unadjusted_close'}

        if l0.isin(fset).sum() >= l1.isin(fset).sum():
            ok = _pick_field(df, 0, ['open'])
            ck = _pick_field(df, 0, ['close', 'unadjusted_close'])
            ak = _pick_field(df, 0, ['adj_close', 'adj close'])
            if ok is not None:
                op = df.xs(ok, axis=1, level=0)
            if ck is not None:
                cl = df.xs(ck, axis=1, level=0)
            if ak is not None:
                adj = df.xs(ak, axis=1, level=0)
        else:
            ok = _pick_field(df, 1, ['open'])
            ck = _pick_field(df, 1, ['close', 'unadjusted_close'])
            ak = _pick_field(df, 1, ['adj_close', 'adj close'])
            if ok is not None:
                op = df.xs(ok, axis=1, level=1)
            if ck is not None:
                cl = df.xs(ck, axis=1, level=1)
            if ak is not None:
                adj = df.xs(ak, axis=1, level=1)

        if cl is None and adj is not None:
            cl = adj.copy()

        if cl is None:
            return None, None, None

        if op is None:
            op = cl.copy()

        cl.columns = [str(c).strip().upper() for c in cl.columns]
        op.columns = [str(c).strip().upper() for c in op.columns]
        if adj is not None:
            adj.columns = [str(c).strip().upper() for c in adj.columns]

    else:
        x = df.copy()
        x.columns = [str(c).strip() for c in x.columns]
        xl = {c: c.lower() for c in x.columns}

        dcol = next((c for c in x.columns if xl[c] in ['date', 'datetime', 'timestamp', 'fecha']), None)
        tcol = next((c for c in x.columns if xl[c] in ['ticker', 'symbol', 'asset', 'activo']), None)
        ocol = next((c for c in x.columns if xl[c] == 'open'), None)
        ccol = next((c for c in x.columns if xl[c] in ['close', 'unadjusted_close']), None)
        acol = next((c for c in x.columns if xl[c] in ['adj_close', 'adj close']), None)

        if dcol is not None and tcol is not None and (ccol is not None or acol is not None):
            cols = [dcol, tcol]
            if ocol is not None:
                cols.append(ocol)
            if ccol is not None:
                cols.append(ccol)
            if acol is not None:
                cols.append(acol)

            y = x[cols].copy()
            y[dcol] = pd.to_datetime(y[dcol], errors='coerce')
            y[tcol] = y[tcol].astype(str).str.upper()
            y = y.dropna(subset=[dcol, tcol])

            if ccol is not None:
                cl = y.pivot_table(index=dcol, columns=tcol, values=ccol, aggfunc='last')
            if acol is not None:
                adj = y.pivot_table(index=dcol, columns=tcol, values=acol, aggfunc='last')
            if cl is None and adj is not None:
                cl = adj.copy()
            if ocol is not None:
                op = y.pivot_table(index=dcol, columns=tcol, values=ocol, aggfunc='last')
            if op is None and cl is not None:
                op = cl.copy()

        else:
            if dcol is not None:
                y = x.copy()
                y[dcol] = pd.to_datetime(y[dcol], errors='coerce')
                y = y.dropna(subset=[dcol]).set_index(dcol)
            elif isinstance(x.index, pd.DatetimeIndex):
                y = x.copy()
            else:
                return None, None, None

            omap, cmap, amap = {}, {}, {}
            for c in y.columns:
                n = str(c).lower().replace('.', '_').replace('-', '_').replace(' ', '_')
                parts = [p for p in n.split('_') if p]
                if len(parts) < 2:
                    continue

                if parts[-1] in ['open', 'close']:
                    fld = parts[-1]
                    tk = '_'.join(parts[:-1]).upper()
                elif parts[0] in ['open', 'close']:
                    fld = parts[0]
                    tk = '_'.join(parts[1:]).upper()
                elif parts[-2:] == ['adj', 'close']:
                    fld = 'adj_close'
                    tk = '_'.join(parts[:-2]).upper()
                elif parts[:2] == ['adj', 'close']:
                    fld = 'adj_close'
                    tk = '_'.join(parts[2:]).upper()
                else:
                    continue

                if fld == 'open':
                    omap[tk] = c
                elif fld == 'close':
                    cmap[tk] = c
                elif fld == 'adj_close':
                    amap[tk] = c

            if len(cmap) == 0 and len(amap) == 0:
                return None, None, None

            idx = y.index
            if len(cmap) > 0:
                cl = pd.DataFrame(index=idx)
                for tk, c in cmap.items():
                    cl[tk] = pd.to_numeric(y[c], errors='coerce')
            if len(amap) > 0:
                adj = pd.DataFrame(index=idx)
                for tk, c in amap.items():
                    adj[tk] = pd.to_numeric(y[c], errors='coerce')
            if cl is None and adj is not None:
                cl = adj.copy()
            if len(omap) > 0:
                op = pd.DataFrame(index=idx)
                for tk, c in omap.items():
                    op[tk] = pd.to_numeric(y[c], errors='coerce')
            if op is None and cl is not None:
                op = cl.copy()

    if cl is None:
        return None, None, None

    def _clean(px):
        if px is None:
            return None
        x = px.copy()
        x.index = pd.to_datetime(x.index, errors='coerce')
        x = x[~x.index.isna()].sort_index()
        x = x[~x.index.duplicated(keep='last')]
        x = x.replace([np.inf, -np.inf], np.nan)
        return x

    op = _clean(op)
    cl = _clean(cl)
    adj = _clean(adj)

    cols = sorted(set(cl.columns) | set(op.columns) | (set(adj.columns) if adj is not None else set()))
    cl = cl.reindex(columns=cols)
    op = op.reindex(index=cl.index, columns=cols)
    if adj is not None:
        adj = adj.reindex(index=cl.index, columns=cols)

    return op, cl, adj


def _adjust_open_close_with_adj(op, cl, adj):
    if adj is None:
        print('Aviso: no se encontro Adj Close. Se usa Close/Open tal cual (asumiendo consistencia).')
        return op, cl, None

    factor = (adj / cl).replace([np.inf, -np.inf], np.nan)

    cl_adj = adj.combine_first(cl)
    op_adj = (op * factor)

    # fallback cuando falta factor puntual
    factor2 = (cl_adj / cl).replace([np.inf, -np.inf], np.nan)
    op_adj = op_adj.combine_first(op * factor2)

    return op_adj, cl_adj, adj


def load_prices():
    # Prioriza ruta heredada de Notebook 1 (si existe en memoria) y rutas explicitas conocidas.
    dynamic_paths = []
    if 'PARQUET_PATH' in globals() and isinstance(globals()['PARQUET_PATH'], str):
        dynamic_paths.append(globals()['PARQUET_PATH'])

    # Fallback robusto: localizar sp500_history.parquet por nombre dentro de Desktop
    # (evita fallos por tildes/encoding en la ruta).
    try:
        infos = fs.LocalFileSystem().get_file_info(fs.FileSelector(r"C:\Users\alons\Desktop", recursive=True))
        for info in infos:
            if info.type == fs.FileType.File and info.path.lower().endswith('sp500_history.parquet'):
                dynamic_paths.append(info.path)
    except Exception:
        pass

    # Mantiene busqueda en carpetas del proyecto para evitar mezclar datasets externos.
    paths = dynamic_paths + PRICE_PATH_HINTS + _scan_files(['data', 'notebooks/data'])

    seen, ordered = set(), []
    for p in paths:
        if p not in seen:
            ordered.append(p)
            seen.add(p)

    best = (None, None, None, None, -1)
    for p in ordered:
        df = _read_any(p)
        op, cl, adj = _norm_prices(df)
        if op is None or cl is None:
            continue

        op_adj, cl_adj, adj_found = _adjust_open_close_with_adj(op, cl, adj)
        score = cl_adj.shape[0] * max(1, cl_adj.shape[1])

        if cl_adj.shape[1] >= TOP_N and cl_adj.shape[0] >= 260:
            print('Fuente precios:', p)
            return op_adj, cl_adj, adj_found, p

        if score > best[4]:
            best = (op_adj, cl_adj, adj_found, p, score)

    if best[0] is not None:
        print('Fuente precios (mejor disponible):', best[3])
        return best[0], best[1], best[2], best[3]

    raise FileNotFoundError('No se pudo leer una fuente de precios valida. Ejecuta Notebook 1 o define PARQUET_PATH a sp500_history.parquet.')


open_px, close_px, adj_close_px, price_source = load_prices()
trading_index = close_px.index.sort_values().unique()

print('Rango fechas (full, incluye warm-up):', trading_index.min(), '->', trading_index.max())
print('#tickers:', close_px.shape[1])
print('Ejemplo columnas:', close_px.columns[:12].tolist())
print('%missing close ajustado (top10):')
display((close_px.isna().mean() * 100).sort_values(ascending=False).head(10).to_frame('missing_%'))


## Carga de selecciones del Notebook 3

In [ ]:
def load_selections():
    paths = SELECTION_PATH_HINTS + _scan_files(["outputs", "notebooks/outputs", ".", "..", "notebooks"])
    seen, ordered = set(), []
    for p in paths:
        if p not in seen:
            ordered.append(p); seen.add(p)

    sel, src = None, None
    for p in ordered:
        if not p.lower().endswith(".csv"):
            continue
        df = _read_any(p)
        if df is None or len(df) == 0:
            continue
        cols = [str(c).strip().lower() for c in df.columns]
        df.columns = cols

        dcol = next((c for c in ["rebalance_date", "date", "fecha"] if c in cols), None)
        tcol = next((c for c in ["ticker", "symbol", "asset", "activo"] if c in cols), None)

        if dcol is not None and tcol is not None:
            x = df.copy()
            x[dcol] = pd.to_datetime(x[dcol], errors="coerce")
            x[tcol] = x[tcol].astype(str).str.upper()
            x = x.dropna(subset=[dcol, tcol])
            if "rank" not in x.columns:
                if "score" in x.columns:
                    x = x.sort_values([dcol, "score", tcol], ascending=[True, False, True])
                else:
                    x = x.sort_values([dcol, tcol], ascending=[True, True])
                x["rank"] = x.groupby(dcol).cumcount() + 1
            keep = [dcol, tcol, "rank"] + [c for c in ["score", "z6", "z12"] if c in x.columns]
            x = x[keep].rename(columns={dcol: "rebalance_date", tcol: "ticker"})
            sel, src = x, p
            break

    if sel is None or len(sel) == 0:
        raise FileNotFoundError("No se pudo cargar el CSV de selecciones (Notebook 3).")

    sel = sel[sel["ticker"] != "GLD"].copy()
    sel = sel.sort_values(["rebalance_date", "rank", "ticker"], ascending=[True, True, True])
    return sel, src


selections_long, selection_source = load_selections()
print("Fuente selecciones:", selection_source)
print("Rango selecciones:", selections_long["rebalance_date"].min(), "->", selections_long["rebalance_date"].max())
print("Fechas unicas:", selections_long["rebalance_date"].nunique())
display(selections_long.head(20))

## Fechas de rebalanceo (intersecci?n calendario real + CSV)

In [ ]:
month_end_dates = get_month_end_dates(trading_index)
month_end_map = pd.Series(month_end_dates, index=month_end_dates.to_period("M"))

s = selections_long.copy()
s["period"] = s["rebalance_date"].dt.to_period("M")
latest_signal = s.groupby("period")["rebalance_date"].max()

selections_by_rebalance = {}
for period, dsel in latest_signal.items():
    if period not in month_end_map.index:
        continue
    dreb = month_end_map.loc[period]
    sub = s[(s["period"] == period) & (s["rebalance_date"] == dsel)].copy()
    sub = sub.sort_values(["rank", "ticker"])
    tickers = []
    for tk in sub["ticker"].tolist():
        if tk not in tickers and tk in close_px.columns:
            tickers.append(tk)
    selections_by_rebalance[dreb] = tickers[:TOP_N]

rebalance_dates = sorted([d for d in selections_by_rebalance.keys() if d in set(month_end_dates) and d in set(trading_index)])

print("#month_end dataset:", len(month_end_dates))
print("#rebalance final:", len(rebalance_dates))
print("Primeras fechas:", rebalance_dates[:8])
if len(rebalance_dates) == 0:
    raise ValueError("No hay fechas de rebalanceo validas.")

## Motor de backtesting (sin look-ahead)

In [ ]:
def _last_close_leq(cl, ticker, d):
    if ticker not in cl.columns:
        return np.nan, pd.NaT, 'NO_PRICE'
    s = cl.loc[:d, ticker].dropna()
    if len(s) == 0:
        return np.nan, pd.NaT, 'NO_PRICE'
    return float(s.iloc[-1]), s.index[-1], 'LAST_CLOSE_PREV'


def _sell_price_exit(op, cl, ticker, d):
    p = op.at[d, ticker] if (d in op.index and ticker in op.columns) else np.nan
    if pd.notna(p) and np.isfinite(p) and p > 0:
        return float(p), d, 'OPEN_D'

    p = cl.at[d, ticker] if (d in cl.index and ticker in cl.columns) else np.nan
    if pd.notna(p) and np.isfinite(p) and p > 0:
        return float(p), d, 'CLOSE_D_FALLBACK'

    return _last_close_leq(cl, ticker, d)


def _close_price(cl, ticker, d):
    p = cl.at[d, ticker] if (d in cl.index and ticker in cl.columns) else np.nan
    if pd.notna(p) and np.isfinite(p) and p > 0:
        return float(p), d, 'CLOSE_D'
    return _last_close_leq(cl, ticker, d)


def _normalize_selections_to_dataset_month_end(selections, trading_index, top_n=20):
    month_ends = get_month_end_dates(trading_index)
    month_end_map = pd.Series(month_ends, index=month_ends.to_period('M'))

    out = {}
    for d, tickers in selections.items():
        d = pd.Timestamp(d)
        period = d.to_period('M')
        if period not in month_end_map.index:
            continue
        d_me = month_end_map.loc[period]

        uniq = []
        for tk in tickers:
            t = str(tk).upper()
            if t == 'GLD':
                continue
            if t not in uniq:
                uniq.append(t)
        out[d_me] = uniq[:top_n]

    return dict(sorted(out.items(), key=lambda x: x[0]))


def _build_monthly_signal_validity(close_full):
    # Validez de senales con lag 1 mes:
    # R6: requiere t-1 y t-7; R12: requiere t-1 y t-13
    month_ends = get_month_end_dates(close_full.index)
    pm = close_full.loc[month_ends].copy()
    valid = pm.notna().shift(1) & pm.notna().shift(7) & pm.notna().shift(13)
    return pm, valid


def _month_id(ts):
    ts = pd.Timestamp(ts)
    return ts.year * 12 + ts.month


def _build_execution_schedule(eligible_rebalances, n_months):
    if len(eligible_rebalances) == 0:
        return []

    n = int(n_months)
    if n <= 1:
        return list(eligible_rebalances)

    out = []
    anchor = eligible_rebalances[0]
    anchor_id = _month_id(anchor)

    for d in eligible_rebalances:
        if (_month_id(d) - anchor_id) % n == 0:
            out.append(d)

    return out


def backtest_momentum_execution(prices_open, prices_close, selections,
                                start_date='2015-01-01', initial_capital=250000.0,
                                top_n=20, fee_rate=0.0023,
                                rebalance_every_n_months=1):
    op_full = prices_open.sort_index().copy()
    cl_full = prices_close.sort_index().copy()

    start_date = pd.Timestamp(start_date)

    trading_index_full = cl_full.index.sort_values().unique()
    selections_norm = _normalize_selections_to_dataset_month_end(selections, trading_index_full, top_n=top_n)
    rebalance_dates_all = sorted(selections_norm.keys())

    pm_full, valid_lag_full = _build_monthly_signal_validity(cl_full)

    valid_score_count = {}
    for d in rebalance_dates_all:
        if d not in valid_lag_full.index:
            valid_score_count[d] = 0
            continue

        row = valid_lag_full.loc[d]
        tgt = selections_norm.get(d, [])
        cnt = 0
        for t in tgt:
            if t in cl_full.columns and d in cl_full.index:
                if bool(row.get(t, False)) and pd.notna(cl_full.at[d, t]):
                    cnt += 1
        valid_score_count[d] = cnt

    first_valid_rebalance = None
    for d in rebalance_dates_all:
        if d >= start_date and valid_score_count.get(d, 0) >= top_n:
            first_valid_rebalance = d
            break

    if first_valid_rebalance is None:
        raise ValueError('No existe rebalanceo valido >= START_DATE con al menos top_n scores validos.')

    eligible_rebalances = [
        d for d in rebalance_dates_all
        if (d >= first_valid_rebalance and valid_score_count.get(d, 0) >= top_n)
    ]
    execution_rebalances = _build_execution_schedule(eligible_rebalances, rebalance_every_n_months)
    execution_set = set(execution_rebalances)

    print('Primer rebalanceo ejecutable (warm-up + score>=top_n):', first_valid_rebalance)
    print('Frecuencia rebalanceo (meses):', rebalance_every_n_months)
    print('Rebalances elegibles:', len(eligible_rebalances), '| Rebalances ejecutados:', len(execution_rebalances))
    print('Resumen score valido por fecha:')
    print(pd.Series(valid_score_count).describe())

    # Reporting y PnL solo desde START_DATE
    dates = trading_index_full[trading_index_full >= start_date]
    op = op_full.reindex(index=dates, columns=cl_full.columns)
    cl = cl_full.reindex(index=dates, columns=cl_full.columns)
    cl_ff = cl.ffill()

    positions = {}
    cash = float(initial_capital)

    eq_rows = []
    tr_rows = []
    wt_rows = []
    fb_rows = []

    prev_date = None

    for d in dates:
        day_turnover = 0.0
        day_fee = 0.0
        day_trades = 0
        did_rebalance = 0

        if d in execution_set:
            row_valid = valid_lag_full.loc[d] if d in valid_lag_full.index else pd.Series(False, index=cl_full.columns)

            target = []
            for t in selections_norm.get(d, []):
                if t not in cl.columns:
                    continue
                if pd.notna(cl.at[d, t]) and bool(row_valid.get(t, False)):
                    target.append(t)
            target = target[:top_n]

            if len(target) == top_n:
                did_rebalance = 1

                old_weights = {}
                aum_prev = cash
                if prev_date is not None:
                    for t, sh in positions.items():
                        if sh <= 0:
                            continue
                        p_prev = cl_ff.at[prev_date, t] if (prev_date in cl_ff.index and t in cl_ff.columns) else np.nan
                        if pd.notna(p_prev) and np.isfinite(p_prev) and p_prev > 0:
                            v = sh * float(p_prev)
                            old_weights[t] = v
                            aum_prev += v

                if aum_prev <= 0:
                    aum_prev = max(cash, 1.0)

                for t in list(old_weights.keys()):
                    old_weights[t] = old_weights[t] / aum_prev

                new_weights = {t: 1.0 / top_n for t in target}
                universe = sorted(set(old_weights.keys()) | set(new_weights.keys()))

                day_turnover = 0.5 * float(np.sum([abs(new_weights.get(t, 0.0) - old_weights.get(t, 0.0)) for t in universe]))
                day_fee = day_turnover * aum_prev * fee_rate

                held = [t for t, sh in positions.items() if sh > 1e-12]
                exits = [t for t in held if t not in set(target)]

                for t in exits:
                    sh = positions.get(t, 0.0)
                    if sh <= 0:
                        continue
                    px, used, src = _sell_price_exit(op, cl, t, d)
                    if not (pd.notna(px) and np.isfinite(px) and px > 0):
                        continue
                    assert pd.Timestamp(used) <= pd.Timestamp(d), 'Look-ahead detectado en precio de salida.'
                    notional = sh * px
                    cash += notional
                    positions[t] = 0.0
                    day_trades += 1
                    tr_rows.append({
                        'date': d, 'ticker': t, 'side': 'SELL',
                        'price': float(px), 'shares': float(sh), 'notional': float(notional),
                        'reason': 'exit', 'price_source': src,
                    })
                    if src != 'OPEN_D':
                        fb_rows.append({'date': d, 'ticker': t, 'side': 'SELL', 'source': src, 'used_price_date': used})

                cash -= day_fee

                cur_values = {}
                for t, sh in positions.items():
                    if sh <= 0:
                        continue
                    px, used, src = _close_price(cl, t, d)
                    if pd.notna(px) and np.isfinite(px) and px > 0:
                        assert pd.Timestamp(used) <= pd.Timestamp(d), 'Look-ahead detectado en marcaje close.'
                        cur_values[t] = sh * px

                eq_for_target = cash + float(np.sum(list(cur_values.values())))
                eq_for_target = max(eq_for_target, 0.0)
                target_value = eq_for_target / top_n

                sell_orders = []
                buy_orders = []
                keys = sorted(set(cur_values.keys()) | set(target))
                for t in keys:
                    desired = target_value if t in target else 0.0
                    current = cur_values.get(t, 0.0)
                    delta = desired - current
                    if abs(delta) < 1e-10:
                        continue
                    px, used, src = _close_price(cl, t, d)
                    if not (pd.notna(px) and np.isfinite(px) and px > 0):
                        continue
                    assert pd.Timestamp(used) <= pd.Timestamp(d), 'Look-ahead detectado en orden close.'
                    if delta < 0:
                        sell_orders.append((t, px, -delta, src, used))
                    else:
                        buy_orders.append((t, px, delta, src, used))

                for t, px, notional_sell, src, used in sell_orders:
                    sh_pos = positions.get(t, 0.0)
                    sh_sell = min(sh_pos, notional_sell / px)
                    if sh_sell <= 0:
                        continue
                    notional = sh_sell * px
                    cash += notional
                    positions[t] = sh_pos - sh_sell
                    day_trades += 1
                    tr_rows.append({
                        'date': d, 'ticker': t, 'side': 'SELL',
                        'price': float(px), 'shares': float(sh_sell), 'notional': float(notional),
                        'reason': 'rebalance', 'price_source': src,
                    })
                    if src != 'CLOSE_D':
                        fb_rows.append({'date': d, 'ticker': t, 'side': 'SELL', 'source': src, 'used_price_date': used})

                total_need = float(np.sum([x[2] for x in buy_orders])) if len(buy_orders) > 0 else 0.0
                scale = 1.0 if (total_need <= cash or total_need <= 0) else max(0.0, cash / total_need)

                for t, px, notional_buy, src, used in buy_orders:
                    n_exec = scale * notional_buy
                    if n_exec <= 0:
                        continue
                    n_exec = min(n_exec, cash)
                    if n_exec <= 0:
                        continue
                    sh_buy = n_exec / px
                    cash -= n_exec
                    positions[t] = positions.get(t, 0.0) + sh_buy
                    day_trades += 1
                    tr_rows.append({
                        'date': d, 'ticker': t, 'side': 'BUY',
                        'price': float(px), 'shares': float(sh_buy), 'notional': float(n_exec),
                        'reason': 'enter' if t not in cur_values else 'rebalance', 'price_source': src,
                    })
                    if src != 'CLOSE_D':
                        fb_rows.append({'date': d, 'ticker': t, 'side': 'BUY', 'source': src, 'used_price_date': used})

                pos_now = {}
                for t, sh in positions.items():
                    if sh <= 0:
                        continue
                    p = cl_ff.at[d, t] if (d in cl_ff.index and t in cl_ff.columns) else np.nan
                    if pd.notna(p) and np.isfinite(p) and p > 0:
                        pos_now[t] = sh * float(p)

                eq_now = cash + float(np.sum(list(pos_now.values())))
                all_tickers = sorted(set(target) | set(pos_now.keys()))
                for t in all_tickers:
                    wt_rows.append({
                        'rebalance_date': d,
                        'ticker': t,
                        'target_weight': (1.0 / top_n) if t in target else 0.0,
                        'achieved_weight': (pos_now.get(t, 0.0) / eq_now) if eq_now > 0 else 0.0,
                        'in_target': int(t in target),
                        'shares': positions.get(t, 0.0),
                        'turnover': day_turnover,
                        'fees_turnover': day_fee,
                        'n_trades_rebalance': day_trades,
                    })

        pos_value = 0.0
        for t, sh in positions.items():
            if sh <= 0:
                continue
            p = cl_ff.at[d, t] if (d in cl_ff.index and t in cl_ff.columns) else np.nan
            if pd.notna(p) and np.isfinite(p) and p > 0:
                pos_value += sh * float(p)

        equity = cash + pos_value
        invested_weight = (pos_value / equity) if equity > 0 else 0.0

        eq_rows.append({
            'date': d,
            'cash': float(cash),
            'positions_value': float(pos_value),
            'equity': float(equity),
            'invested_weight_sum': float(invested_weight),
            'turnover': float(day_turnover),
            'fees_turnover': float(day_fee),
            'n_trades': int(day_trades),
            'did_rebalance': int(did_rebalance),
        })

        prev_date = d

    equity_df = pd.DataFrame(eq_rows).set_index('date').sort_index()
    trades_df = pd.DataFrame(tr_rows)
    weights_df = pd.DataFrame(wt_rows)
    fallback_df = pd.DataFrame(fb_rows)

    max_w = equity_df['invested_weight_sum'].max()
    print('max(sum(weights)) <= 1+1e-8  ->', max_w)
    assert max_w <= 1.0 + 1e-8, 'Apalancamiento accidental detectado: sum(weights) > 1'

    daily_ret = equity_df['equity'].pct_change().dropna()
    extreme = daily_ret[np.abs(daily_ret) > 0.30]
    print('N retornos diarios con |ret|>0.3:', len(extreme))
    if len(extreme) > 0:
        print('Extremos (revisar si hubo eventos excepcionales):')
        print(extreme.sort_values(key=np.abs, ascending=False).head(10))

    pre_turn = equity_df.loc[equity_df.index < first_valid_rebalance, 'turnover'].sum()
    print('turnover acumulado antes de primer rebalanceo valido:', pre_turn)
    assert np.isclose(pre_turn, 0.0), 'Turnover no es cero antes del primer rebalanceo valido.'

    non_exec_turn = equity_df.loc[~equity_df.index.isin(execution_rebalances), 'turnover'].sum()
    assert np.isclose(non_exec_turn, 0.0), 'Hay turnover fuera del calendario de rebalanceo configurado.'

    assert equity_df.index.min() >= start_date, 'La curva incluye fechas anteriores al inicio de reporting.'
    print('Curva/m?tricas recortadas a >=', start_date.date())

    n_reb_exec = int(equity_df['did_rebalance'].sum())
    n_trades_total = int(equity_df['n_trades'].sum())
    avg_trades_reb = (n_trades_total / n_reb_exec) if n_reb_exec > 0 else 0.0

    print('Rebalances ejecutados reales:', n_reb_exec)
    print('Trades totales:', n_trades_total)
    print('Trades promedio por rebalanceo:', round(avg_trades_reb, 2))

    summary = {
        'equity_final': float(equity_df['equity'].iloc[-1]),
        'total_fees': float(equity_df['fees_turnover'].sum()),
        'n_trades': n_trades_total,
        'turnover_total': float(equity_df['turnover'].sum()),
        'first_valid_rebalance': first_valid_rebalance,
        'n_rebalances_executed': n_reb_exec,
        'rebalance_every_n_months': int(rebalance_every_n_months),
    }

    return equity_df, trades_df, weights_df, fallback_df, summary


equity_df, trades_df, weights_df, fallback_df, summary = backtest_momentum_execution(
    open_px, close_px, selections_by_rebalance,
    start_date=START_DATE,
    initial_capital=INITIAL_CAPITAL,
    top_n=TOP_N,
    fee_rate=FEE_RATE,
    rebalance_every_n_months=REBALANCE_EVERY_N_MONTHS,
)

print('Equity final:', round(summary['equity_final'], 2))
print('Total fees:', round(summary['total_fees'], 2))
print('#trades:', summary['n_trades'])
print('Turnover total:', round(summary['turnover_total'], 6))
print('Primer rebalanceo valido:', summary['first_valid_rebalance'])
print('Rebalances ejecutados:', summary['n_rebalances_executed'])
print('Frecuencia (meses):', summary['rebalance_every_n_months'])


## Benchmark, metricas y outputs

In [ ]:
if BENCH in close_px.columns:
    bench_close = close_px[BENCH].ffill()
else:
    s = equity_df.index.min().strftime("%Y-%m-%d")
    e = (equity_df.index.max() + pd.Timedelta(days=3)).strftime("%Y-%m-%d")
    spy = yf.download(BENCH, start=s, end=e, progress=False, auto_adjust=False)
    if isinstance(spy.columns, pd.MultiIndex):
        if ("Close", BENCH) in spy.columns:
            bench_close = spy[("Close", BENCH)]
        elif ("Adj Close", BENCH) in spy.columns:
            bench_close = spy[("Adj Close", BENCH)]
        else:
            bench_close = spy.xs("Close", axis=1, level=0).iloc[:, 0]
    else:
        bench_close = spy["Close"] if "Close" in spy.columns else spy["Adj Close"]

bench_close.index = pd.to_datetime(bench_close.index)
bench_close = bench_close.reindex(equity_df.index).ffill()

metrics = compute_metrics(equity_df["equity"], bench_close)
print("Metricas:")
for k, v in metrics.items():
    print(k, v)

lfs = fs.LocalFileSystem()
out_saved = []
for out_dir in ["outputs", "notebooks/outputs"]:
    try:
        lfs.create_dir(out_dir, recursive=True)
        p1 = f"{out_dir}/backtest_equity_curve.csv"
        p2 = f"{out_dir}/trades.csv"
        p3 = f"{out_dir}/weights_by_rebalance.csv"
        p4 = f"{out_dir}/fallback_price_log.csv"
        equity_df.to_csv(p1)
        trades_df.to_csv(p2, index=False)
        weights_df.to_csv(p3, index=False)
        fallback_df.to_csv(p4, index=False)
        out_saved = [p1, p2, p3, p4]
        break
    except Exception:
        pass

if len(out_saved) == 0:
    equity_df.to_csv("backtest_equity_curve.csv")
    trades_df.to_csv("trades.csv", index=False)
    weights_df.to_csv("weights_by_rebalance.csv", index=False)
    fallback_df.to_csv("fallback_price_log.csv", index=False)
    out_saved = ["backtest_equity_curve.csv", "trades.csv", "weights_by_rebalance.csv", "fallback_price_log.csv"]

print("Outputs:")
for p in out_saved:
    print(p)

In [ ]:
eq = equity_df["equity"].copy()
bench_norm = bench_close / bench_close.dropna().iloc[0] * eq.iloc[0]

plt.figure(figsize=(12, 5))
plt.plot(eq.index, eq.values, label="Strategy")
plt.plot(bench_norm.index, bench_norm.values, label="SPY normalizado")
plt.title("Equity Curve")
plt.legend()
plt.tight_layout()
plt.show()

dd = eq / eq.cummax() - 1
plt.figure(figsize=(12, 3.8))
plt.plot(dd.index, dd.values, color="firebrick")
plt.title("Drawdown")
plt.tight_layout()
plt.show()

## Analisis critico (resumen)
- La comisi?n m?nima de 23$ penaliza m?s a ?rdenes peque?as.
- Puede haber sesgo de supervivencia seg?n el universo heredado de NB1/NB2.
- El motor evita look-ahead usando solo OPEN/CLOSE de D o ?ltimo CLOSE <= D.
- No se modela slippage ni impacto de mercado (limitaci?n).

## Debug profundo de escalas (splits) + fix y re-backtest


In [ ]:
# Celda 1: snapshot baseline
baseline_equity_df = equity_df.copy()
baseline_trades_df = trades_df.copy()
baseline_weights_df = weights_df.copy()
baseline_summary = dict(summary)

print('Baseline equity final:', round(float(baseline_equity_df['equity'].iloc[-1]), 2))
print('Baseline total fees:', round(float(baseline_summary.get('total_fees', np.nan)), 2))
print('Baseline n_trades:', int(len(baseline_trades_df)))


In [ ]:
# Celda 2: utilidades robustas para extraer OHLC + unadjusted_close desde la fuente original

def _canon(c):
    return str(c).strip().lower().replace(" ", "_").replace("-", "_").replace(".", "_")

def _find_col(cols, candidates):
    cl = {_canon(c): c for c in cols}
    for cand in candidates:
        if cand in cl:
            return cl[cand]
    return None

def _read_source_table(path):
    p = str(path).lower()
    if p.endswith(".parquet"):
        return pq.read_table(path).to_pandas()
    if p.endswith(".csv"):
        return pd.read_csv(path)
    raise ValueError(f"Formato no soportado: {path}")

def _extract_field(df, field):
    # MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        l0 = pd.Index([_canon(x) for x in df.columns.get_level_values(0)])
        l1 = pd.Index([_canon(x) for x in df.columns.get_level_values(1)])
        keys = [field]
        if field == "unadjusted_close":
            keys += ["unadjusted close"]
        if field == "close":
            keys += ["adj_close", "adj close"]

        for k in keys:
            k2 = _canon(k)
            if k2 in l0.values:
                out = df.xs(k2, axis=1, level=0)
                out.columns = [str(c).upper() for c in out.columns]
                return out
            if k2 in l1.values:
                out = df.xs(k2, axis=1, level=1)
                out.columns = [str(c).upper() for c in out.columns]
                return out

    # Long format
    date_col = _find_col(df.columns, ["date", "datetime", "timestamp", "fecha"])
    ticker_col = _find_col(df.columns, ["ticker", "symbol", "asset", "activo"])
    field_col = _find_col(df.columns, [field, field.replace("_", " ")])
    if field_col is None and field == "close":
        field_col = _find_col(df.columns, ["adj_close", "adj close", "close"])
    if date_col and ticker_col and field_col:
        x = df[[date_col, ticker_col, field_col]].copy()
        x[date_col] = pd.to_datetime(x[date_col], errors="coerce")
        x[ticker_col] = x[ticker_col].astype(str).str.upper()
        x = x.dropna(subset=[date_col, ticker_col, field_col])
        out = x.pivot_table(index=date_col, columns=ticker_col, values=field_col, aggfunc="last")
        return out.sort_index()

    # Flat wide columns
    x = df.copy()
    if not isinstance(x.index, pd.DatetimeIndex):
        dcol = _find_col(x.columns, ["date", "datetime", "timestamp", "fecha"])
        if dcol is None:
            return None
        x[dcol] = pd.to_datetime(x[dcol], errors="coerce")
        x = x.dropna(subset=[dcol]).set_index(dcol)

    out = pd.DataFrame(index=x.index)
    for c in x.columns:
        n = _canon(c)
        parts = [p for p in n.split("_") if p]
        if len(parts) < 2:
            continue

        tk = None
        if field in ["open", "high", "low", "close"]:
            if parts[-1] == field:
                tk = "_".join(parts[:-1]).upper()
            elif parts[0] == field:
                tk = "_".join(parts[1:]).upper()
            elif field == "close" and parts[-2:] in (["adj", "close"],):
                tk = "_".join(parts[:-2]).upper()
        elif field == "unadjusted_close":
            if parts[-2:] == ["unadjusted", "close"]:
                tk = "_".join(parts[:-2]).upper()

        if tk and len(tk) > 0:
            out[tk] = pd.to_numeric(x[c], errors="coerce")

    if out.shape[1] == 0:
        return None
    return out.sort_index()

def load_ohlc_from_price_source(price_source_path):
    raw = _read_source_table(price_source_path)

    open_w = _extract_field(raw, "open")
    high_w = _extract_field(raw, "high")
    low_w = _extract_field(raw, "low")
    close_w = _extract_field(raw, "close")
    unadj_w = _extract_field(raw, "unadjusted_close")

    if open_w is None or close_w is None or unadj_w is None:
        raise ValueError("No pude extraer open/close/unadjusted_close. Revisa formato del parquet.")

    idx = pd.DatetimeIndex(sorted(set(open_w.index) | set(close_w.index) | set(unadj_w.index)))
    cols = sorted(set(open_w.columns) | set(close_w.columns) | set(unadj_w.columns))

    open_w = open_w.reindex(index=idx, columns=cols)
    close_w = close_w.reindex(index=idx, columns=cols)
    unadj_w = unadj_w.reindex(index=idx, columns=cols)

    if high_w is not None:
        high_w = high_w.reindex(index=idx, columns=cols)
    if low_w is not None:
        low_w = low_w.reindex(index=idx, columns=cols)

    return open_w, high_w, low_w, close_w, unadj_w

open_raw_w, high_raw_w, low_raw_w, close_w, unadj_w = load_ohlc_from_price_source(price_source)

print("Shape OPEN:", open_raw_w.shape, "CLOSE:", close_w.shape, "UNADJ:", unadj_w.shape)
print("Rango fechas:", close_w.index.min(), "->", close_w.index.max())


In [ ]:
# Celda 3: diagnostico global de mismatch de escalas
adj_factor = (close_w / unadj_w).replace([np.inf, -np.inf], np.nan)
r_open_raw = (open_raw_w / unadj_w).replace([np.inf, -np.inf], np.nan)
r_open_close = (open_raw_w / close_w).replace([np.inf, -np.inf], np.nan)

q = [0.001, 0.01, 0.5, 0.99, 0.999]

print("Quantiles adj_factor = CLOSE / UNADJ_CLOSE")
print(adj_factor.stack().dropna().quantile(q))
print("\nQuantiles r_open_raw = OPEN / UNADJ_CLOSE")
print(r_open_raw.stack().dropna().quantile(q))
print("\nQuantiles r_open_close = OPEN / CLOSE")
print(r_open_close.stack().dropna().quantile(q))

top_adj = adj_factor.stack().rename("adj_factor").to_frame()
top_adj["abs_adj_minus_1"] = (top_adj["adj_factor"] - 1.0).abs()
top_adj = top_adj.sort_values("abs_adj_minus_1", ascending=False).head(30)

top_roc = r_open_close.stack().rename("r_open_close").to_frame()
top_roc["abs_log"] = np.abs(np.log(top_roc["r_open_close"].replace(0, np.nan)))
top_roc = top_roc.replace([np.inf, -np.inf], np.nan).dropna().sort_values("abs_log", ascending=False).head(30)

print("\nTOP 30 |adj_factor-1|")
display(top_adj)
print("TOP 30 extremos r_open_close")
display(top_roc)

med_r_open_raw = float(r_open_raw.stack().dropna().median())
q_roc = r_open_close.stack().dropna().quantile(q)

mix_confirmed = (
    abs(med_r_open_raw - 1.0) < 0.10 and
    (
        float((adj_factor.stack().dropna() - 1.0).abs().quantile(0.99)) > 0.20
        or q_roc.loc[0.999] > 2.0
        or q_roc.loc[0.001] < 0.5
    )
)

print("\nmediana r_open_raw:", med_r_open_raw)
print("Mismatch de escalas confirmado:", mix_confirmed)


In [ ]:
# Celda 4: diagnostico por trades y saltos de equity
def diagnose_trades_and_jumps(eq_df, tr_df, open_df, close_df, unadj_df, focus_date="2021-01-29"):
    tr = tr_df.copy()
    tr["date"] = pd.to_datetime(tr["date"])
    eq = eq_df.copy()
    eq.index = pd.to_datetime(eq.index)

    rebalance_dates_exec = set(eq.index[eq["did_rebalance"] == 1])
    outside = tr[~tr["date"].isin(rebalance_dates_exec)].copy()

    print("Dias con trades:", len(pd.Index(tr["date"].unique())))
    print("Trades fuera de rebalance_dates:", len(outside))

    eq_ret = eq["equity"].pct_change()
    top10 = eq_ret.abs().sort_values(ascending=False).head(10)
    print("\nTOP 10 |equity_ret|:")
    display(eq_ret.loc[top10.index].sort_values(key=np.abs, ascending=False).to_frame("equity_ret"))

    for d in top10.index:
        tr_d = tr[tr["date"] == d].copy()
        print(f"\n===== Fecha salto {d.date()} =====")
        print("n_trades:", len(tr_d))
        if len(tr_d) > 0:
            display(tr_d[["date", "ticker", "side", "notional", "price", "price_source"]].sort_values("notional", ascending=False))

            tks = sorted(tr_d["ticker"].unique().tolist())
            md = pd.DataFrame(index=tks)
            md["open"] = open_df.loc[d, tks].values if d in open_df.index else np.nan
            md["close"] = close_df.loc[d, tks].values if d in close_df.index else np.nan
            md["unadj_close"] = unadj_df.loc[d, tks].values if d in unadj_df.index else np.nan
            md["open_over_close"] = md["open"] / md["close"]
            md["close_over_unadj"] = md["close"] / md["unadj_close"]
            display(md.sort_values("close_over_unadj", ascending=False))

    fd = pd.Timestamp(focus_date)
    print(f"\n===== Fecha sospechosa {fd.date()} =====")
    tr_f = tr[tr["date"] == fd].copy()
    print("n_trades:", len(tr_f))
    if len(tr_f) > 0:
        display(tr_f[["date", "ticker", "side", "notional", "price", "price_source"]].sort_values("notional", ascending=False))
        tks = sorted(tr_f["ticker"].unique().tolist())
        md = pd.DataFrame(index=tks)
        md["open"] = open_df.loc[fd, tks].values if fd in open_df.index else np.nan
        md["close"] = close_df.loc[fd, tks].values if fd in close_df.index else np.nan
        md["unadj_close"] = unadj_df.loc[fd, tks].values if fd in unadj_df.index else np.nan
        md["open_over_close"] = md["open"] / md["close"]
        md["close_over_unadj"] = md["close"] / md["unadj_close"]
        display(md.sort_values("close_over_unadj", ascending=False))
    else:
        print("No hubo trades en esa fecha.")

diagnose_trades_and_jumps(
    baseline_equity_df,
    baseline_trades_df,
    open_raw_w,
    close_w,
    unadj_w,
    focus_date="2021-01-29"
)


In [ ]:
# Celda 5: FIX de escalas + asserts
if mix_confirmed:
    adj_factor_fix = (close_w / unadj_w).replace([np.inf, -np.inf], np.nan)
    open_adj_w = (open_raw_w * adj_factor_fix).replace([np.inf, -np.inf], np.nan)
    high_adj_w = (high_raw_w * adj_factor_fix).replace([np.inf, -np.inf], np.nan) if high_raw_w is not None else None
    low_adj_w = (low_raw_w * adj_factor_fix).replace([np.inf, -np.inf], np.nan) if low_raw_w is not None else None
    close_adj_w = close_w.copy()
    print("Fix aplicado: OPEN/HIGH/LOW ajustados con adj_factor; CLOSE ajustado como referencia.")
else:
    open_adj_w = open_raw_w.copy()
    high_adj_w = high_raw_w.copy() if high_raw_w is not None else None
    low_adj_w = low_raw_w.copy() if low_raw_w is not None else None
    close_adj_w = close_w.copy()
    print("No se confirma mismatch fuerte. Mantengo escala original y revisa otras causas.")

ratio_scale = (open_adj_w / close_adj_w).stack().replace([np.inf, -np.inf], np.nan).dropna()
ratio_q = ratio_scale.quantile([0.001, 0.01, 0.5, 0.99, 0.999])
print("\nQuantiles OPEN_ADJ/CLOSE_ADJ")
print(ratio_q)

assert ratio_q.loc[0.999] < 1.5, "p99.9 OPEN_ADJ/CLOSE_ADJ demasiado alto."
assert ratio_q.loc[0.001] > 0.67, "p0.1 OPEN_ADJ/CLOSE_ADJ demasiado bajo."

asset_ret = close_adj_w.pct_change()
extreme_list = (
    asset_ret.stack()
    .rename("ret")
    .to_frame()
    .assign(abs_ret=lambda x: x["ret"].abs())
    .query("abs_ret > 0.5")
    .sort_values("abs_ret", ascending=False)
    .head(20)
)
print("\nTop 20 retornos diarios por activo |ret|>0.5")
display(extreme_list)


In [ ]:
# Celda 6: re-ejecutar backtest con precios corregidos
cols_bt = close_px.columns if "close_px" in globals() else close_adj_w.columns
open_bt = open_adj_w.reindex(columns=cols_bt)
close_bt = close_adj_w.reindex(columns=cols_bt)

equity_fix_df, trades_fix_df, weights_fix_df, fallback_fix_df, summary_fix = backtest_momentum_execution(
    open_bt, close_bt, selections_by_rebalance,
    start_date=START_DATE,
    initial_capital=INITIAL_CAPITAL,
    top_n=TOP_N,
    fee_rate=FEE_RATE,
    rebalance_every_n_months=REBALANCE_EVERY_N_MONTHS,
)

print("FIX equity final:", round(float(summary_fix["equity_final"]), 2))
print("FIX total fees:", round(float(summary_fix["total_fees"]), 2))
print("FIX n_trades:", int(summary_fix["n_trades"]))
print("FIX turnover total:", round(float(summary_fix["turnover_total"]), 6))


In [ ]:
# Celda 7: comparacion before/after + top 10 saltos + graficos
def _max_dd(eq):
    dd = eq / eq.cummax() - 1.0
    return float(dd.min())

cmp = pd.DataFrame(
    [
        {
            "equity_final": float(baseline_equity_df["equity"].iloc[-1]),
            "max_drawdown": _max_dd(baseline_equity_df["equity"]),
            "n_trades": int(len(baseline_trades_df)),
            "total_fees": float(baseline_summary.get("total_fees", np.nan)),
            "turnover_total": float(baseline_summary.get("turnover_total", np.nan)),
        },
        {
            "equity_final": float(equity_fix_df["equity"].iloc[-1]),
            "max_drawdown": _max_dd(equity_fix_df["equity"]),
            "n_trades": int(len(trades_fix_df)),
            "total_fees": float(summary_fix.get("total_fees", np.nan)),
            "turnover_total": float(summary_fix.get("turnover_total", np.nan)),
        },
    ],
    index=["before", "after_fix"]
)
print("Comparacion before vs after_fix")
display(cmp)

ret_b = baseline_equity_df["equity"].pct_change()
ret_a = equity_fix_df["equity"].pct_change()
top10_dates = ret_b.abs().sort_values(ascending=False).head(10).index

tb = baseline_trades_df.copy()
tb["date"] = pd.to_datetime(tb["date"])
ta = trades_fix_df.copy()
ta["date"] = pd.to_datetime(ta["date"])

cmp_jumps = pd.DataFrame({
    "ret_before": ret_b.reindex(top10_dates),
    "ret_after": ret_a.reindex(top10_dates),
    "n_trades_before": [int((tb["date"] == d).sum()) for d in top10_dates],
    "n_trades_after": [int((ta["date"] == d).sum()) for d in top10_dates],
    "notional_before": [float(tb.loc[tb["date"] == d, "notional"].sum()) for d in top10_dates],
    "notional_after": [float(ta.loc[ta["date"] == d, "notional"].sum()) for d in top10_dates],
}).sort_values("ret_before", key=np.abs, ascending=False)

print("\nTop 10 saltos de equity before vs after_fix")
display(cmp_jumps)

if "SPY" in close_bt.columns:
    bench_close = close_bt["SPY"].ffill()
else:
    s = equity_fix_df.index.min().strftime("%Y-%m-%d")
    e = (equity_fix_df.index.max() + pd.Timedelta(days=3)).strftime("%Y-%m-%d")
    spy = yf.download("SPY", start=s, end=e, progress=False, auto_adjust=False)
    if isinstance(spy.columns, pd.MultiIndex):
        if ("Adj Close", "SPY") in spy.columns:
            bench_close = spy[("Adj Close", "SPY")]
        elif ("Close", "SPY") in spy.columns:
            bench_close = spy[("Close", "SPY")]
        else:
            bench_close = spy.xs("Close", axis=1, level=0).iloc[:, 0]
    else:
        bench_close = spy["Adj Close"] if "Adj Close" in spy.columns else spy["Close"]

bench_close.index = pd.to_datetime(bench_close.index)
bench_close = bench_close.reindex(equity_fix_df.index).ffill()

eq_b = baseline_equity_df["equity"].reindex(equity_fix_df.index).ffill()
eq_a = equity_fix_df["equity"].copy()
spy_norm = bench_close / bench_close.dropna().iloc[0] * eq_a.iloc[0]

plt.figure(figsize=(12, 5))
plt.plot(eq_b.index, eq_b.values, label="Strategy BEFORE", alpha=0.7)
plt.plot(eq_a.index, eq_a.values, label="Strategy AFTER FIX", linewidth=2)
plt.plot(spy_norm.index, spy_norm.values, label="SPY normalizado", alpha=0.8)
plt.title("Equity Curve: Before vs After split-fix")
plt.legend()
plt.tight_layout()
plt.show()

dd_b = eq_b / eq_b.cummax() - 1.0
dd_a = eq_a / eq_a.cummax() - 1.0

plt.figure(figsize=(12, 4))
plt.plot(dd_b.index, dd_b.values, label="DD BEFORE", alpha=0.7)
plt.plot(dd_a.index, dd_a.values, label="DD AFTER FIX", linewidth=2)
plt.title("Drawdown: Before vs After split-fix")
plt.legend()
plt.tight_layout()
plt.show()


## Resumen corto
- Este bloque detecta mismatch entre OPEN y CLOSE usando UNADJUSTED_CLOSE.
- Si se confirma, corrige OPEN/HIGH/LOW a escala de CLOSE y re-ejecuta el backtest.
- Compara before/after con m?tricas, saltos de equity y gr?ficos.
